In [47]:
from llava.model.builder import load_pretrained_model
import argparse
from llava.mm_utils import tokenizer_image_token, process_images, get_model_name_from_path

model_path = "llava-fastvithd_0.5b_stage3"
model_name = get_model_name_from_path(model_path)

parser = argparse.ArgumentParser()
parser.add_argument("--model-path", type=str, default="./llava-v1.5-13b")
parser.add_argument("--model-base", type=str, default=None)
parser.add_argument("--image-file", type=str, default=None, help="location of image file")
parser.add_argument("--prompt", type=str, default="Describe the image.", help="Prompt for VLM.")
parser.add_argument("--conv-mode", type=str, default="mistral")
parser.add_argument("--temperature", type=float, default=0.2)
parser.add_argument("--top_p", type=float, default=None)
parser.add_argument("--num_beams", type=int, default=1)
args, unknown = parser.parse_known_args()

tokenizer, model, image_processor, context_len = load_pretrained_model(
    model_path, args.model_base, model_name, device="cuda"
)

In [48]:
device = "cuda"

In [49]:
llm_model = model.model
print(llm_model)

LlavaQwen2Model(
  (embed_tokens): Embedding(151646, 896)
  (layers): ModuleList(
    (0-23): 24 x Qwen2DecoderLayer(
      (self_attn): Qwen2Attention(
        (q_proj): Linear(in_features=896, out_features=896, bias=True)
        (k_proj): Linear(in_features=896, out_features=128, bias=True)
        (v_proj): Linear(in_features=896, out_features=128, bias=True)
        (o_proj): Linear(in_features=896, out_features=896, bias=False)
      )
      (mlp): Qwen2MLP(
        (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
        (up_proj): Linear(in_features=896, out_features=4864, bias=False)
        (down_proj): Linear(in_features=4864, out_features=896, bias=False)
        (act_fn): SiLUActivation()
      )
      (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
    )
  )
  (norm): Qwen2RMSNorm((896,), eps=1e-06)
  (rotary_emb): Qwen2RotaryEmbedding()
  (vision_tower): MobileCLIPVisionTower(
    (vi

In [23]:
llm_model.vision_tower = None
llm_model.mm_projector = None

In [51]:
model.model.vision_tower

MobileCLIPVisionTower(
  (vision_tower): MCi(
    (model): FastViT(
      (patch_embed): Sequential(
        (0): MobileOneBlock(
          (se): Identity()
          (activation): GELU(approximate='none')
          (reparam_conv): Conv2d(3, 96, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
        )
        (1): MobileOneBlock(
          (se): Identity()
          (activation): GELU(approximate='none')
          (reparam_conv): Conv2d(96, 96, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), groups=96)
        )
        (2): MobileOneBlock(
          (se): Identity()
          (activation): GELU(approximate='none')
          (reparam_conv): Conv2d(96, 96, kernel_size=(1, 1), stride=(1, 1))
        )
      )
      (network): ModuleList(
        (0): Sequential(
          (0): RepMixerBlock(
            (token_mixer): RepMixer(
              (reparam_conv): Conv2d(96, 96, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=96)
            )
            (convffn): ConvFFN(
  

In [24]:
llm_model

LlavaQwen2Model(
  (embed_tokens): Embedding(151646, 896)
  (layers): ModuleList(
    (0-23): 24 x Qwen2DecoderLayer(
      (self_attn): Qwen2Attention(
        (q_proj): Linear(in_features=896, out_features=896, bias=True)
        (k_proj): Linear(in_features=896, out_features=128, bias=True)
        (v_proj): Linear(in_features=896, out_features=128, bias=True)
        (o_proj): Linear(in_features=896, out_features=896, bias=False)
      )
      (mlp): Qwen2MLP(
        (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
        (up_proj): Linear(in_features=896, out_features=4864, bias=False)
        (down_proj): Linear(in_features=4864, out_features=896, bias=False)
        (act_fn): SiLUActivation()
      )
      (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
    )
  )
  (norm): Qwen2RMSNorm((896,), eps=1e-06)
  (rotary_emb): Qwen2RotaryEmbedding()
  (vision_tower): None
  (mm_projector): None
)

In [25]:
import torch

# Disable ALL SDPA kernels
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(False)
torch.backends.cuda.enable_math_sdp(True)

In [26]:
# llm_model.config._attn_implementation = "eager"

In [27]:
# llm_model.config.num_key_value_heads = llm_model.config.num_attention_heads

In [28]:
from transformers.models.qwen2 import modeling_qwen2

def dummy_causal_mask(*args, **kwargs):
    return None

modeling_qwen2.create_causal_mask = dummy_causal_mask

In [29]:
class ONNXWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, input_ids):
        inputs_embeds = self.model.embed_tokens(input_ids)

        batch_size, seq_len = input_ids.shape
        position_ids = torch.arange(seq_len, device=input_ids.device)\
            .unsqueeze(0).expand(batch_size, -1)

        outputs = self.model(
            inputs_embeds=inputs_embeds,
            attention_mask=None,
            position_ids=position_ids,
            use_cache=False,
            return_dict=False
        )

        return outputs[0]

wrapped_model = ONNXWrapper(llm_model).to(device)
wrapped_model.eval()

ONNXWrapper(
  (model): LlavaQwen2Model(
    (embed_tokens): Embedding(151646, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2

In [30]:
# Dummy inputs
batch_size = 1
seq_len = 4
input_ids = torch.randint(0, 151646, (batch_size, seq_len), dtype=torch.long, device=device)
attention_mask = torch.ones((batch_size, seq_len), dtype=torch.long, device=device)

torch.onnx.export(
    wrapped_model,
    (input_ids,),   # 🔥 only one input now

    "llava_qwen2_real.onnx",

    input_names=["input_ids"],
    output_names=["last_hidden_state"],

    dynamic_axes={
        "input_ids": {0: "batch", 1: "seq"},
        "last_hidden_state": {0: "batch", 1: "seq"}
    },

    opset_version=17
)

C:\Users\kumar\AppData\Local\Programs\Python\Python312\Lib\site-packages\transformers\integrations\sdpa_attention.py:81: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  is_causal = query.shape[2] > 1 and attention_mask is None and getattr(module, "is_causal", True)


AssertionError: conversion of scaled_dot_product_attention not implemented if enable_gqa is True

In [19]:
wrapped_model

ONNXWrapper(
  (model): LlavaQwen2Model(
    (embed_tokens): Embedding(151646, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2

In [39]:
import onnxruntime as ort
import numpy as np
from PIL import Image
import torchvision.transforms as transforms

vision_session = ort.InferenceSession("vision.onnx")

# preprocess image
image = Image.open("GOT.jpg").convert("RGB")

preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

img = preprocess(image).unsqueeze(0).numpy()

vision_out = vision_session.run(None, {"pixel_values": img})

image_features = vision_out[0]   # shape: (1, N, D)

AttributeError: module 'onnxruntime' has no attribute 'InferenceSession'

In [42]:
import platform
print(platform.architecture())

('64bit', 'WindowsPE')


In [41]:
import onnxruntime as ort
print(ort.__version__)

AttributeError: module 'onnxruntime' has no attribute '__version__'

In [35]:
import onnxruntime as ort

# GPU session
session = ort.InferenceSession("llm.onnx", providers=['CUDAExecutionProvider'])
print("Available providers:", session.get_providers())

AttributeError: module 'onnxruntime' has no attribute 'InferenceSession'

In [ ]:
tokenizer, model, image_processor, context_len = load_pretrained_model(model_path, args.model_base, model_name, device="cuda")


tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2")

text = "Describe this image:"
input_ids = tokenizer(text, return_tensors="np")["input_ids"]